# Третья лабораторная работа

Подготовка обучающей и тестовой выборки, кросс-валидация и подбор гиперпараметров на примере метода ближайших соседей.

В данной лабораторной работе используется набор данных German Credit Risk, загружаемый через интерфейс OpenML. Для реализации алгоритма K-ближайших соседей, предобработки данных и оценки качества моделей применяются инструменты библиотеки scikit-learn.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

Загружаем набор данных credit-g. В датасете присутствуют как числовые, так и категориальные признаки. Поскольку метод KNN основан на вычислении расстояний между объектами, необходимо выполнить два шага предобработки:

1. Закодировать категориальные признаки (например, с помощью One-Hot Encoding).

2. Масштабировать данные, чтобы признаки с большими значениями (например, сумма кредита) не доминировали над признаками с малыми значениями (например, возраст).

In [2]:
# Загрузка данных
data = fetch_openml(name='credit-g', version=1, as_frame=True, parser='auto')
df = data.frame

# Проверка на наличие пропусков
print("Количество пропусков:\n", df.isnull().sum().sum())

df.info()

Количество пропусков:
 0
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   checking_status         1000 non-null   category
 1   duration                1000 non-null   int64   
 2   credit_history          1000 non-null   category
 3   purpose                 1000 non-null   category
 4   credit_amount           1000 non-null   int64   
 5   savings_status          1000 non-null   category
 6   employment              1000 non-null   category
 7   installment_commitment  1000 non-null   int64   
 8   personal_status         1000 non-null   category
 9   other_parties           1000 non-null   category
 10  residence_since         1000 non-null   int64   
 11  property_magnitude      1000 non-null   category
 12  age                     1000 non-null   int64   
 13  other_payment_plans     1000 non-null   category
 14  housing    

Кодируем признаки:

In [3]:
# Разделение на признаки и целевую переменную
X = df.drop('class', axis=1)
y = df['class']

# Кодирование признаков
X_encoded = pd.get_dummies(X, drop_first=True)

# Кодируем целевую переменную в 1 и 0
y_encoded = y.map({'bad': 0, 'good': 1})

# Вывод первых строк
display(X_encoded.head())

,duration,credit_amount,installment_commitment,residence_since,age,existing_credits,num_dependents,checking_status_<0,checking_status_>=200,checking_status_no checking,...,property_magnitude_car,other_payment_plans_none,other_payment_plans_stores,housing_own,housing_rent,job_unemp/unskilled non res,job_unskilled resident,job_skilled,own_telephone_yes,foreign_worker_yes
0,6,1169,4,4,67,2,1,True,False,False,...,False,True,False,True,False,False,False,True,True,True
1,48,5951,2,2,22,1,1,False,False,False,...,False,True,False,True,False,False,False,True,False,True
2,12,2096,2,3,49,1,2,False,False,True,...,False,True,False,True,False,False,True,False,False,True
3,42,7882,2,4,45,1,2,True,False,False,...,False,True,False,False,False,False,False,True,False,True
4,24,4870,3,4,53,2,2,True,False,False,...,False,True,False,False,False,False,False,True,False,True


Разделяем датасет на обучающую (80%) и тестовую (20%) выборки с использованием train_test_split. Параметр stratify=y гарантирует, что баланс классов сохранится в обеих выборках. Затем применяем StandardScaler к обучающей и тестовой выборкам.

In [4]:
# Разделение выборки 
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Масштабирование признаков
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Размер обучающей выборки: {X_train_scaled.shape}")
print(f"Размер тестовой выборки: {X_test_scaled.shape}")

Размер обучающей выборки: (800, 48)
Размер тестовой выборки: (200, 48)


Обучим исходную модель K-ближайших соседей для произвольно заданного гиперпараметра K (например, n_neighbors = 5). Оценим качество с помощью метрики Accuracy и отчета о классификации.

In [5]:
# Инициализация и обучение модели с K=5
clf_base = KNeighborsClassifier(n_neighbors=5)
clf_base.fit(X_train_scaled, y_train)

# Предсказание и оценка
y_pred_base = clf_base.predict(X_test_scaled)

print("Метрики для базовой модели (K = 5):\n")
print(classification_report(y_test, y_pred_base))

Метрики для базовой модели (K = 5):

              precision    recall  f1-score   support

           0       0.47      0.32      0.38        60
           1       0.74      0.85      0.79       140

    accuracy                           0.69       200
   macro avg       0.61      0.58      0.59       200
weighted avg       0.66      0.69      0.67       200



Для поиска оптимального количества соседей (K) и метрики расстояния (Евклидова или Манхэттенская) применим GridSearchCV и RandomizedSearchCV.
Используем две стратегии кросс-валидации:

1. StratifiedKFold — классическое разбиение на 5 фолдов.

2. StratifiedShuffleSplit — случайное перемешивание и разбиение на 5 независимых сплитов.

In [10]:
# Сетка параметров
param_grid = {
    'n_neighbors': np.arange(1, 600),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

# Стратегия 1: StratifiedKFold + GridSearchCV
cv_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    KNeighborsClassifier(), 
    param_grid, 
    cv=cv_stratified, 
    scoring='accuracy', 
    n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train)

# Стратегия 2: StratifiedShuffleSplit + RandomizedSearchCV
cv_shuffle = StratifiedShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
random_search = RandomizedSearchCV(
    KNeighborsClassifier(), 
    param_distributions=param_grid, 
    n_iter=15,
    cv=cv_shuffle, 
    scoring='accuracy', 
    n_jobs=-1,
    random_state=42
)
random_search.fit(X_train_scaled, y_train)

print(f"Лучшие параметры (GridSearchCV): {grid_search.best_params_}")
print(f"Accuracy на кросс-валидации (GridSearchCV): {grid_search.best_score_:.4f}\n")

print(f"Лучшие параметры (RandomizedSearchCV): {random_search.best_params_}")
print(f"Accuracy на кросс-валидации (RandomSearch): {random_search.best_score_:.4f}")

Лучшие параметры (GridSearchCV): {'metric': 'manhattan', 'n_neighbors': np.int64(26), 'weights': 'uniform'}
Accuracy на кросс-валидации (GridSearchCV): 0.7450

Лучшие параметры (RandomizedSearchCV): {'weights': 'uniform', 'n_neighbors': np.int64(21), 'metric': 'manhattan'}
Accuracy на кросс-валидации (RandomSearch): 0.7350


Оценим лучшую модель, найденную с помощью GridSearchCV, на отложенной тестовой выборке и сравним её с результатами базовой модели (K=5).

In [7]:
# Извлечение лучшей модели
best_knn = grid_search.best_estimator_

# Предсказание на тестовой выборке
y_pred_best = best_knn.predict(X_test_scaled)

print("Сравнение результатов на ТЕСТОВОЙ выборке:\n")
print("--- Базовая модель (K=5) ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_base):.4f}\n")

print(f"--- Оптимальная модель {grid_search.best_params_} ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_best):.4f}\n")

print("Детальный отчет для ОПТИМАЛЬНОЙ модели:")
print(classification_report(y_test, y_pred_best))

Сравнение результатов на ТЕСТОВОЙ выборке:

--- Базовая модель (K=5) ---
Accuracy: 0.6900

--- Оптимальная модель {'metric': 'manhattan', 'n_neighbors': np.int64(26), 'weights': 'uniform'} ---
Accuracy: 0.7500

Детальный отчет для ОПТИМАЛЬНОЙ модели:
              precision    recall  f1-score   support

           0       0.69      0.30      0.42        60
           1       0.76      0.94      0.84       140

    accuracy                           0.75       200
   macro avg       0.73      0.62      0.63       200
weighted avg       0.74      0.75      0.71       200

